In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Introducing Z3

In this notebook we are going to solve the following exercise using the constraint solver `Z3`.

   * I have as many brothers as I have sisters, but
   * my sister has twice as many brothers as she has sisters.

How many children does my father have?

First, we load the <samp>OCaml</samp> binding of Z3.  It is installed via `opam install z3`.  The file `z3_helpers.ml` loads this library and defines a few functions that make it more convenient to use.  The comment at the beginning of this file gives an overview of these functions.

In [ ]:
import_file "../z3_helpers.ml"

All Z3 objects belong to a *context*.  The file `z3_helpers.ml` defines the context `ctx`, which we use for all our objects.

In [ ]:
ctx

In `Z3`, every variable has to be declared.  In this case, all our variables are integer valued.  The function `int_const name` declares an integer variable with the given name.  It calls the function `Z3.Arithmetic.Integer.mk_const_s` of the Z3 library.

The value `int_const "boys"` is the `Z3` representation of a constraint variable.  We assign this value to the variable `boys`.  Its type is `Z3.Expr.expr`, the type of all Z3 expressions.

In [ ]:
let boys  = int_const "boys"
let girls = int_const "girls"

Next, we create a *solver* object.  This is a constraint solver capable of solving constraint satisfaction problems.  The function `new_solver ()` calls `Z3.Solver.mk_solver ctx None`.

In [ ]:
let s = new_solver ()

We can add constraints to this solver object via the function `Z3.Solver.add`, which takes a solver and a list of constraints.
We will use two variables:
 - `boys` is the number of my father's sons,
 - `girls` is the number of my father's daughters.

Note that the number of my brothers is `boys - 1` as I am not my own brother.  Similarly, the number of my sister's sisters is `girls - 1`.

We write the constraints as strings.  The function `constraints`, which is defined in the file `z3_helpers.ml`, takes a list of strings and turns each of them into a Z3 expression.  In these strings, a variable is written as the name that was given to `int_const`.  The strings use the usual notation of formulas:
- the arithmetic operators `+`, `-`, `*`, `/`, and `mod`,
- the comparisons `=`, `<>`, `<`, `<=`, `>`, and `>=`, and
- the logical operators `&&`, `||`, `not`, `=>` (implication), and `<=>` (equivalence).

Instead of these operators, the symbols `≠`, `≤`, `≥`, `∧`, `∨`, `¬`, `→`, and `↔` can be used, too.  The comment at the beginning of the file `z3_helpers.ml` describes the syntax in detail.

So the statement that I have as many brothers as sisters, but my sister has twice as many brothers as she has sisters is formalized as follows:

In [ ]:
Z3.Solver.add s (constraints ["boys - 1 = girls"; "2 * (girls - 1) = boys"])

The solver now contains the following constraints, which are shown in the notation of Z3:

In [ ]:
s

The function `Z3.Solver.check` examines whether the given set of constraints is satisfiable.  Its second argument is a list of additional assumptions, which we do not need.

In general, it can return one of the following results:
- `SATISFIABLE` is returned if the problem is solvable,
- `UNSATISFIABLE` is returned if the problem is unsatisfiable,
- `UNKNOWN` is returned if the constraint solver is not powerful enough to solve the given problem.

In [ ]:
Z3.Solver.check s []

To extract the solution of the given problem we use the function `solver_model`.  It calls `Z3.Solver.get_model`, which returns `Some model` after a successful check.

In [ ]:
let model = solver_model s

In order to extract the values for `boys` and `girls`, we use the function `eval_int model x`.  It evaluates the expression `x` in the model and converts the result into an <samp>OCaml</samp> integer.

In [ ]:
let b = eval_int model boys
let g = eval_int model girls

In [ ]:
Printf.printf "My father has %d children: %d boys and %d girls.\n" (b + g) b g;
Printf.printf "I have %d - 1 = %d brothers and %d sisters.\n" b (b - 1) g;
Printf.printf "My sister has %d - 1 = %d sisters and %d = 2 * %d brothers.\n" g (g - 1) b (g - 1)